In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!java --version

openjdk 11.0.28 2025-07-15
OpenJDK Runtime Environment (build 11.0.28+6-post-Ubuntu-1ubuntu122.04.1)
OpenJDK 64-Bit Server VM (build 11.0.28+6-post-Ubuntu-1ubuntu122.04.1, mixed mode, sharing)


In [3]:
!pip list | grep pyspark

pyspark                               3.5.1


In [4]:
# Remove old Java
!apt-get remove openjdk-* -y

# Install OpenJDK 17
!apt-get update -q
!apt-get install openjdk-17-jdk -y

# Set JAVA_HOME
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Verify
!java -version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'openjdk-11-jdk' for glob 'openjdk-*'
Note, selecting 'openjdk-11-jre' for glob 'openjdk-*'
Note, selecting 'openjdk-11-jre-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-19-jre-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-8-jre-zero' for glob 'openjdk-*'
Note, selecting 'openjdk-11-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-21-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-19-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-21-demo' for glob 'openjdk-*'
Note, selecting 'openjdk-18-jdk-headless' for glob 'openjdk-*'
Note, selecting 'openjdk-17-dbg' for glob 'openjdk-*'
Note, selecting 'openjdk-17-doc' for glob 'openjdk-*'
Note, selecting 'openjdk-18-dbg' for glob 'openjdk-*'
Note, selecting 'openjdk-17-jdk' for glob 'openjdk-*'
Note, selecting 'openjdk-18-doc' for glob 'openjdk-*'
Note, selecting 'openjdk-17-jre' f

In [5]:
!pip install --upgrade pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 13.4 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813800 sha256=29abfd096452cd138f8cc2e7a5dcf7cf1b4b8fb6d1146655726d4b123e2d1565
  Stored in directory: /root/.cache/pip/wheels/31/9f/68/f89fb34ccd886909be7d0e390eaaf97f21efdf540c0ee8dbcd
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.1
    Uninstalling pyspark-3.5.1:
      Successfully uninstalled pyspark-3.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [6]:
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T

from pyspark.sql import SparkSession

# Não limitar a largura das colunas apresentadas
pd.options.display.max_colwidth = None
# Não usar a notação científica (ex: 6.125000e-02) e usar 6 casas decimais (ex: 0.061250)
pd.options.display.float_format = "{:.6f}".format
# Não utilizar matplotlib como engine de gráficos e usar plotly
pd.options.plotting.backend = "plotly"

In [7]:
# Criando um cluster local com 1 executor e a quantidade de threads igual a quantidade de cores de CPU disponíveis

spark = SparkSession.builder\
  .master("local[*]")\
  .config("spark.executor.memory", "4g")\
  .config("spark.driver.memory", "4g")\
  .getOrCreate()
spark

In [8]:
driver_memory = spark.conf.get("spark.driver.memory")
print(f"Driver memory: {driver_memory}")

Driver memory: 4g


In [9]:
executor_memory = spark.conf.get("spark.executor.memory")
print(f"Executor memory: {executor_memory}")

Executor memory: 4g


In [41]:
# Comando para desativar os recursos do spark
#spark.stop()

## Explorando os datasets

In [11]:
ROOT_DATA_PATH = "drive/MyDrive/data/ml-25m"

In [12]:
movies_df = spark.read.csv(f"{ROOT_DATA_PATH}/movies.csv", header=True, inferSchema=True)
movies_df.show(5)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


In [13]:
type(movies_df)

pyspark.sql.classic.dataframe.DataFrame

In [14]:
movies_df.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



In [15]:
tags_df = spark.read.csv(f"{ROOT_DATA_PATH}/tags.csv", header=True, inferSchema=True)
tags_df.show(5)

+------+-------+----------------+----------+
|userId|movieId|             tag| timestamp|
+------+-------+----------------+----------+
|     3|    260|         classic|1439472355|
|     3|    260|          sci-fi|1439472256|
|     4|   1732|     dark comedy|1573943598|
|     4|   1732|  great dialogue|1573943604|
|     4|   7569|so bad it's good|1573943455|
+------+-------+----------------+----------+
only showing top 5 rows


In [16]:
tags_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: string (nullable = true)



In [17]:
tags_df = tags_df.withColumn('timestamp', F.to_timestamp(F.from_unixtime('timestamp')))
tags_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [18]:
tags_df.show(5)

+------+-------+----------------+-------------------+
|userId|movieId|             tag|          timestamp|
+------+-------+----------------+-------------------+
|     3|    260|         classic|2015-08-13 13:25:55|
|     3|    260|          sci-fi|2015-08-13 13:24:16|
|     4|   1732|     dark comedy|2019-11-16 22:33:18|
|     4|   1732|  great dialogue|2019-11-16 22:33:24|
|     4|   7569|so bad it's good|2019-11-16 22:30:55|
+------+-------+----------------+-------------------+
only showing top 5 rows


In [19]:
ratings_df = spark.read.csv(f"{ROOT_DATA_PATH}/ratings.csv", header=True, inferSchema=True)\
  .withColumn('timestamp', F.to_timestamp(F.from_unixtime('timestamp')))
ratings_df.show(5)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|    296|   5.0|2006-05-17 15:34:04|
|     1|    306|   3.5|2006-05-17 12:26:57|
|     1|    307|   5.0|2006-05-17 12:27:08|
|     1|    665|   5.0|2006-05-17 15:13:40|
|     1|    899|   3.5|2006-05-17 12:21:50|
+------+-------+------+-------------------+
only showing top 5 rows


In [20]:
ratings_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [21]:
links_df = spark.read.csv(f"{ROOT_DATA_PATH}/links.csv", header=True, inferSchema=True)
links_df.show(5)

+-------+------+------+
|movieId|imdbId|tmdbId|
+-------+------+------+
|      1|114709|   862|
|      2|113497|  8844|
|      3|113228| 15602|
|      4|114885| 31357|
|      5|113041| 11862|
+-------+------+------+
only showing top 5 rows


In [22]:
links_df.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- imdbId: integer (nullable = true)
 |-- tmdbId: integer (nullable = true)



In [23]:
gtags_df = spark.read.csv(f"{ROOT_DATA_PATH}/genome-tags.csv", header=True, inferSchema=True)
gtags_df.show(5)

+-----+------------+
|tagId|         tag|
+-----+------------+
|    1|         007|
|    2|007 (series)|
|    3|18th century|
|    4|       1920s|
|    5|       1930s|
+-----+------------+
only showing top 5 rows


In [24]:
gtags_df.printSchema()

root
 |-- tagId: integer (nullable = true)
 |-- tag: string (nullable = true)



In [25]:
gscores_df = spark.read.csv(f"{ROOT_DATA_PATH}/genome-scores.csv", header=True, inferSchema=True)
gscores_df.show(5)

+-------+-----+--------------------+
|movieId|tagId|           relevance|
+-------+-----+--------------------+
|      1|    1|0.028749999999999998|
|      1|    2|0.023749999999999993|
|      1|    3|              0.0625|
|      1|    4| 0.07574999999999998|
|      1|    5|             0.14075|
+-------+-----+--------------------+
only showing top 5 rows


In [26]:
gscores_df.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- tagId: integer (nullable = true)
 |-- relevance: double (nullable = true)



## Executando merge de dados

In [27]:
movies_df.count()

62423

In [28]:
links_df.count()

62423

In [29]:
movies_df = movies_df.join(links_df, on='movieId', how='inner')
movies_df = movies_df.cache()
movies_df.show(5)

+-------+--------------------+--------------------+------+------+
|movieId|               title|              genres|imdbId|tmdbId|
+-------+--------------------+--------------------+------+------+
|      1|    Toy Story (1995)|Adventure|Animati...|114709|   862|
|      2|      Jumanji (1995)|Adventure|Childre...|113497|  8844|
|      3|Grumpier Old Men ...|      Comedy|Romance|113228| 15602|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|114885| 31357|
|      5|Father of the Bri...|              Comedy|113041| 11862|
+-------+--------------------+--------------------+------+------+
only showing top 5 rows


In [30]:
movies_df.count()

62423

## Quais são os top 10 filmes mais avaliados?

In [31]:
ratings_df.show(5)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|    296|   5.0|2006-05-17 15:34:04|
|     1|    306|   3.5|2006-05-17 12:26:57|
|     1|    307|   5.0|2006-05-17 12:27:08|
|     1|    665|   5.0|2006-05-17 15:13:40|
|     1|    899|   3.5|2006-05-17 12:21:50|
+------+-------+------+-------------------+
only showing top 5 rows


In [32]:
%%time

ratings_df.count()

CPU times: user 1.92 ms, sys: 1.29 ms, total: 3.21 ms
Wall time: 10.8 s


25000095

In [33]:
%%time

df = ratings_df.groupBy('movieId')\
  .count()\
  .withColumnRenamed('count', 'ratings_count')\
  .orderBy(F.desc('ratings_count'))\
  .limit(10)\
  .cache() # Para salvar o df computado na memória RAM dos workers do cluster. Sai imediatamente.

CPU times: user 4.28 ms, sys: 1.07 ms, total: 5.36 ms
Wall time: 147 ms


In [34]:
%%time

df.show(5)

+-------+-------------+
|movieId|ratings_count|
+-------+-------------+
|    356|        81491|
|    318|        81482|
|    296|        79672|
|    593|        74127|
|   2571|        72674|
+-------+-------------+
only showing top 5 rows
CPU times: user 10 ms, sys: 3.04 ms, total: 13.1 ms
Wall time: 50.2 s


In [35]:
%%time

df.show(5)

+-------+-------------+
|movieId|ratings_count|
+-------+-------------+
|    356|        81491|
|    318|        81482|
|    296|        79672|
|    593|        74127|
|   2571|        72674|
+-------+-------------+
only showing top 5 rows
CPU times: user 1.23 ms, sys: 37 µs, total: 1.26 ms
Wall time: 164 ms


In [36]:
df.toPandas()

,movieId,ratings_count
0,356,81491
1,318,81482
2,296,79672
3,593,74127
4,2571,72674
5,260,68717
6,480,64144
7,527,60411
8,110,59184
9,2959,58773


In [37]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

In [38]:
type(df.toPandas())

pandas.core.frame.DataFrame

In [39]:
movies_df.select('movieId', 'title').toPandas()

,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)
...,...,...
62418,209157,We (2018)
62419,209159,Window of the Soul (2001)
62420,209163,Bad Poems (2018)
62421,209169,A Girl Thing (2001)


In [ ]:
%%time

# Merge ruim acontecendo no driver program ("client")

df.toPandas()\
  .merge(
      movies_df.select('movieId', 'title').toPandas(), # Potencial estouro de memória no driver program
      on='movieId',
      how='inner'
  )

CPU times: user 551 ms, sys: 1.85 ms, total: 552 ms
Wall time: 1.5 s


,movieId,ratings_count,title
0,356,81491,Forrest Gump (1994)
1,318,81482,"Shawshank Redemption, The (1994)"
2,296,79672,Pulp Fiction (1994)
3,593,74127,"Silence of the Lambs, The (1991)"
4,2571,72674,"Matrix, The (1999)"
5,260,68717,Star Wars: Episode IV - A New Hope (1977)
6,480,64144,Jurassic Park (1993)
7,527,60411,Schindler's List (1993)
8,110,59184,Braveheart (1995)
9,2959,58773,Fight Club (1999)


In [40]:
%%time

# Merge ideal acontecendo no cluster

df = movies_df.select('movieId', 'title')\
  .join(df, on='movieId', how='inner')\
  .orderBy(F.desc('ratings_count'))
df.toPandas()

CPU times: user 14.8 ms, sys: 2.83 ms, total: 17.6 ms
Wall time: 1.15 s


,movieId,title,ratings_count
0,356,Forrest Gump (1994),81491
1,318,"Shawshank Redemption, The (1994)",81482
2,296,Pulp Fiction (1994),79672
3,593,"Silence of the Lambs, The (1991)",74127
4,2571,"Matrix, The (1999)",72674
5,260,Star Wars: Episode IV - A New Hope (1977),68717
6,480,Jurassic Park (1993),64144
7,527,Schindler's List (1993),60411
8,110,Braveheart (1995),59184
9,2959,Fight Club (1999),58773


## Atividade Turma

### Quais são os top 10 filmes com maior total da soma das avaliações?

In [ ]:
%%time

df_task = ratings_df.groupBy('movieId')\
  .sum('rating')\
  .withColumnRenamed('sum(rating)', 'ratings_sum')\
  .orderBy(F.desc('ratings_sum'))\
  .limit(10)\
  .cache()
df_task.toPandas()

CPU times: user 18 ms, sys: 5.73 ms, total: 23.7 ms
Wall time: 49.4 s


,movieId,ratings_sum
0,318,359627.0
1,296,333739.0
2,356,329876.5
3,593,307726.5
4,2571,301895.0
5,260,283127.0
6,527,256600.5
7,2959,248510.5
8,1196,237711.0
9,50,237207.5


In [ ]:
df_task = movies_df.select('movieId', 'title')\
  .join(df_task, on='movieId', how='inner')\
  .orderBy(F.desc('ratings_sum'))
df_task.toPandas()

,movieId,title,ratings_sum
0,318,"Shawshank Redemption, The (1994)",359627.0
1,296,Pulp Fiction (1994),333739.0
2,356,Forrest Gump (1994),329876.5
3,593,"Silence of the Lambs, The (1991)",307726.5
4,2571,"Matrix, The (1999)",301895.0
5,260,Star Wars: Episode IV - A New Hope (1977),283127.0
6,527,Schindler's List (1993),256600.5
7,2959,Fight Club (1999),248510.5
8,1196,Star Wars: Episode V - The Empire Strikes Back...,237711.0
9,50,"Usual Suspects, The (1995)",237207.5


## Qual é a quantidade de avaliações de cada valor de avaliação dos top 10 filmes mais avaliados?

In [ ]:
ratings_df.limit(5).toPandas()

,userId,movieId,rating,timestamp
0,1,296,5.0,2006-05-17 15:34:04
1,1,306,3.5,2006-05-17 12:26:57
2,1,307,5.0,2006-05-17 12:27:08
3,1,665,5.0,2006-05-17 15:13:40
4,1,899,3.5,2006-05-17 12:21:50


In [ ]:
df.show()

+-------+--------------------+-------------+
|movieId|               title|ratings_count|
+-------+--------------------+-------------+
|    356| Forrest Gump (1994)|        81491|
|    318|Shawshank Redempt...|        81482|
|    296| Pulp Fiction (1994)|        79672|
|    593|Silence of the La...|        74127|
|   2571|  Matrix, The (1999)|        72674|
|    260|Star Wars: Episod...|        68717|
|    480|Jurassic Park (1993)|        64144|
|    527|Schindler's List ...|        60411|
|    110|   Braveheart (1995)|        59184|
|   2959|   Fight Club (1999)|        58773|
+-------+--------------------+-------------+



In [ ]:
ratings_df.count()

25000095

In [ ]:
df.select(F.collect_list('movieId')).first()['collect_list(movieId)']

[356, 318, 296, 593, 2571, 260, 480, 527, 110, 2959]

In [ ]:
df.select(F.collect_list('movieId')).first()[0]

[356, 318, 296, 593, 2571, 260, 480, 527, 110, 2959]

In [ ]:
ratings_df.where(F.col('movieId').isin(df.select(F.collect_list('movieId')).first()[0])).count()

700675

In [ ]:
%%time

df = ratings_df.where(F.col('movieId').isin(df.select(F.collect_list('movieId')).first()[0]))\
  .groupBy('movieId', 'rating')\
  .count()\
  .withColumnRenamed('count', 'ratings_value_count')\
  .join(df, on='movieId', how='inner')\
  .orderBy(F.desc('ratings_count'), F.desc('ratings_value_count'))\
  .cache()
df.where("movieId = 356").toPandas()

CPU times: user 22.1 ms, sys: 5.55 ms, total: 27.6 ms
Wall time: 31 s


,movieId,rating,ratings_value_count,title,ratings_count
0,356,5.0,25918,Forrest Gump (1994),81491
1,356,4.0,23348,Forrest Gump (1994),81491
2,356,3.0,10380,Forrest Gump (1994),81491
3,356,4.5,9609,Forrest Gump (1994),81491
4,356,3.5,6185,Forrest Gump (1994),81491
5,356,2.0,2449,Forrest Gump (1994),81491
6,356,2.5,1569,Forrest Gump (1994),81491
7,356,1.0,1159,Forrest Gump (1994),81491
8,356,1.5,450,Forrest Gump (1994),81491
9,356,0.5,424,Forrest Gump (1994),81491


In [ ]:
df.count()

100

## Atividade Turma

### Quais são os top 10 filmes com maior quantidade de avaliações com valor 5?

In [ ]:
%%time

df_task = ratings_df.where(F.col('rating') == 5)\
  .groupBy('movieId')\
  .count()\
  .withColumnRenamed('count', 'count_rating_5')\
  .orderBy(F.desc('count_rating_5'))\
  .limit(10)
df_task.toPandas()

CPU times: user 14.9 ms, sys: 3.25 ms, total: 18.2 ms
Wall time: 37.3 s


,movieId,count_rating_5
0,318,39553
1,296,32169
2,356,25918
3,260,25804
4,2571,25482
5,527,24853
6,593,24801
7,858,24418
8,50,21585
9,2959,21486


In [ ]:
%%time

df_task = movies_df.select('movieId', 'title')\
  .join(df_task, on='movieId', how='inner')\
  .orderBy(F.desc('count_rating_5'))
df_task.toPandas()

CPU times: user 9.9 ms, sys: 6.09 ms, total: 16 ms
Wall time: 32.8 s


,movieId,title,count_rating_5
0,318,"Shawshank Redemption, The (1994)",39553
1,296,Pulp Fiction (1994),32169
2,356,Forrest Gump (1994),25918
3,260,Star Wars: Episode IV - A New Hope (1977),25804
4,2571,"Matrix, The (1999)",25482
5,527,Schindler's List (1993),24853
6,593,"Silence of the Lambs, The (1991)",24801
7,858,"Godfather, The (1972)",24418
8,50,"Usual Suspects, The (1995)",21585
9,2959,Fight Club (1999),21486


## Quais são os top 10 filmes mais avaliados que são do gênero "Children"?

In [ ]:
movies_df.limit(5).toPandas()

,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357
4,5,Father of the Bride Part II (1995),Comedy,113041,11862


In [ ]:
movies_df.where(F.col('genres').like("%Thriller%")).limit(5).toPandas()

,movieId,title,genres,imdbId,tmdbId
0,6,Heat (1995),Action|Crime|Thriller,113277,949
1,10,GoldenEye (1995),Action|Adventure|Thriller,113189,710
2,20,Money Train (1995),Action|Comedy|Crime|Drama|Thriller,113845,11517
3,21,Get Shorty (1995),Comedy|Crime|Thriller,113161,8012
4,22,Copycat (1995),Crime|Drama|Horror|Mystery|Thriller,112722,1710


In [ ]:
%%time

movies_df.where(F.col('genres').like("%Children%"))\
  .join(ratings_df, on='movieId', how='inner')\
  .groupBy('title')\
  .count()\
  .orderBy(F.desc('count'))\
  .limit(10)\
  .toPandas()

CPU times: user 15.2 ms, sys: 4.18 ms, total: 19.4 ms
Wall time: 31.3 s


,title,count
0,Toy Story (1995),57309
1,Aladdin (1992),43387
2,"Lion King, The (1994)",42745
3,Shrek (2001),42303
4,Beauty and the Beast (1991),35723
5,Finding Nemo (2003),34712
6,E.T. the Extra-Terrestrial (1982),34602
7,"Monsters, Inc. (2001)",34572
8,Babe (1995),31456
9,"Incredibles, The (2004)",30562


In [ ]:
%%time

ratings_df.join(movies_df.select('movieId', 'title', 'genres'), on='movieId', how='inner')\
    .where("genres LIKE '%Children%'")\
    .groupBy('title')\
    .count()\
    .orderBy(F.desc('count'))\
    .limit(10)\
    .toPandas()

CPU times: user 15.6 ms, sys: 3.52 ms, total: 19.1 ms
Wall time: 32 s


,title,count
0,Toy Story (1995),57309
1,Aladdin (1992),43387
2,"Lion King, The (1994)",42745
3,Shrek (2001),42303
4,Beauty and the Beast (1991),35723
5,Finding Nemo (2003),34712
6,E.T. the Extra-Terrestrial (1982),34602
7,"Monsters, Inc. (2001)",34572
8,Babe (1995),31456
9,"Incredibles, The (2004)",30562


## Quantos filmes cada gênero possui?

In [ ]:
movies_df.limit(5).toPandas()

,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357
4,5,Father of the Bride Part II (1995),Comedy,113041,11862


In [ ]:
movies_df.select(F.split('genres', '[|]').alias('col')).limit(5).toPandas()

,col
0,"[Adventure, Animation, Children, Comedy, Fantasy]"
1,"[Adventure, Children, Fantasy]"
2,"[Comedy, Romance]"
3,"[Comedy, Drama, Romance]"
4,[Comedy]


In [ ]:
movies_df.select(F.explode(F.split('genres', '[|]'))).limit(8).toPandas()

,col
0,Adventure
1,Animation
2,Children
3,Comedy
4,Fantasy
5,Adventure
6,Children
7,Fantasy


In [ ]:
movies_df.select(F.explode(F.split('genres', '[|]')))\
  .groupBy('col')\
  .count()\
  .orderBy(F.desc('count'))\
  .withColumnRenamed('col', 'genre')\
  .toPandas()\
  .set_index('genre')\
  .plot(kind='bar')

## Atividade Turma

### Quais são as médias de avaliações dos 10 filmes mais avaliados do gênero Drama? Ordene crescente pelas médias.

In [ ]:
%%time

drama_movies_df = movies_df.where(F.col('genres').contains("Drama"))
drama_movies_df.count()

CPU times: user 2.22 ms, sys: 424 µs, total: 2.64 ms
Wall time: 319 ms


25606

In [ ]:
%%time

drama_movies_ratings_df = ratings_df.join(drama_movies_df, on='movieId', how='inner')
drama_movies_ratings_df.count()

CPU times: user 5.9 ms, sys: 117 µs, total: 6.02 ms
Wall time: 20.3 s


10962833

In [ ]:
drama_movies_ratings_df.limit(3).toPandas()

,movieId,userId,rating,timestamp,title,genres,imdbId,tmdbId
0,296,1,5.0,2006-05-17 15:34:04,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,110912,680
1,306,1,3.5,2006-05-17 12:26:57,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,111495,110
2,307,1,5.0,2006-05-17 12:27:08,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama,108394,108


In [ ]:
%%time

df_task = drama_movies_ratings_df.groupBy('movieId')\
  .count()\
  .withColumnRenamed('count', 'ratings_count')\
  .orderBy(F.desc('ratings_count'))\
  .limit(10)\
  .cache()
df_task.toPandas()

CPU times: user 11.3 ms, sys: 2.83 ms, total: 14.1 ms
Wall time: 26.3 s


,movieId,ratings_count
0,356,81491
1,318,81482
2,296,79672
3,527,60411
4,110,59184
5,2959,58773
6,2858,53689
7,858,52498
8,7153,50797
9,150,48377


In [ ]:
top_10_drama = df_task.toPandas()['movieId'].tolist()
top_10_drama

[356, 318, 296, 527, 110, 2959, 2858, 858, 7153, 150]

In [ ]:
%%time

df_result = drama_movies_ratings_df.where(F.col('movieId').isin(top_10_drama))\
  .groupBy('title')\
  .mean('rating')\
  .withColumnRenamed('avg(rating)', 'ratings_mean')\
  .orderBy(F.asc('ratings_mean'))\
  .cache()
df_result.toPandas()

CPU times: user 16.1 ms, sys: 2.28 ms, total: 18.4 ms
Wall time: 20 s


,title,ratings_mean
0,Apollo 13 (1995),3.873556
1,Braveheart (1995),4.002273
2,Forrest Gump (1994),4.048011
3,"Lord of the Rings: The Return of the King, The...",4.090340
4,American Beauty (1999),4.107340
5,Pulp Fiction (1994),4.188912
6,Fight Club (1999),4.228311
7,Schindler's List (1993),4.247579
8,"Godfather, The (1972)",4.324336
9,"Shawshank Redemption, The (1994)",4.413576


## Qual é a estatística da variável 'relevance' do genoma de tag?

In [ ]:
gscores_df.show(5)

+-------+-----+--------------------+
|movieId|tagId|           relevance|
+-------+-----+--------------------+
|      1|    1|0.028749999999999998|
|      1|    2|0.023749999999999993|
|      1|    3|              0.0625|
|      1|    4| 0.07574999999999998|
|      1|    5|             0.14075|
+-------+-----+--------------------+
only showing top 5 rows


In [ ]:
gscores_df.count()

15584448

In [ ]:
gscores_df.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- tagId: integer (nullable = true)
 |-- relevance: double (nullable = true)



In [ ]:
%%time

df = gscores_df.describe('relevance').toPandas()
df

CPU times: user 6.55 ms, sys: 2.56 ms, total: 9.11 ms
Wall time: 17.5 s


,summary,relevance
0,count,15584448
1,mean,0.11636786379922436
2,stddev,0.15447224288787412
3,min,2.4999999999997247E-4
4,max,1.0


## Quais são as top 5 tags mais relevantes do filme 'Matrix, The (1999)' segundo o genoma de tags?

In [ ]:
movies_df.show(5)

+-------+--------------------+--------------------+------+------+
|movieId|               title|              genres|imdbId|tmdbId|
+-------+--------------------+--------------------+------+------+
|      1|    Toy Story (1995)|Adventure|Animati...|114709|   862|
|      2|      Jumanji (1995)|Adventure|Childre...|113497|  8844|
|      3|Grumpier Old Men ...|      Comedy|Romance|113228| 15602|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|114885| 31357|
|      5|Father of the Bri...|              Comedy|113041| 11862|
+-------+--------------------+--------------------+------+------+
only showing top 5 rows


In [ ]:
movie_name = "Matrix, The (1999)"
movie_id = movies_df.where(F.col('title') == movie_name)\
  .collect()[0]['movieId']
movie_id

2571

In [ ]:
gscores_df.show(3)

+-------+-----+--------------------+
|movieId|tagId|           relevance|
+-------+-----+--------------------+
|      1|    1|0.028749999999999998|
|      1|    2|0.023749999999999993|
|      1|    3|              0.0625|
+-------+-----+--------------------+
only showing top 3 rows


In [ ]:
gtags_df.show(3)

+-----+------------+
|tagId|         tag|
+-----+------------+
|    1|         007|
|    2|007 (series)|
|    3|18th century|
+-----+------------+
only showing top 3 rows


In [ ]:
%%time

tag_genome_df = gscores_df.join(gtags_df, on='tagId', how='inner')\
  .cache()
tag_genome_df.show(5)

+-----+-------+--------------------+------------+
|tagId|movieId|           relevance|         tag|
+-----+-------+--------------------+------------+
|    1|      1|0.028749999999999998|         007|
|    2|      1|0.023749999999999993|007 (series)|
|    3|      1|              0.0625|18th century|
|    4|      1| 0.07574999999999998|       1920s|
|    5|      1|             0.14075|       1930s|
+-----+-------+--------------------+------------+
only showing top 5 rows
CPU times: user 8.45 ms, sys: 2.32 ms, total: 10.8 ms
Wall time: 39 s


In [ ]:
tag_genome_df

DataFrame[tagId: int, movieId: int, relevance: double, tag: string]

In [ ]:
tag_genome_df.where(F.col('movieId') == movie_id)\
  .orderBy(F.desc('relevance'))\
  .limit(5)\
  .show()

+-----+-------+------------------+---------------+
|tagId|movieId|         relevance|            tag|
+-----+-------+------------------+---------------+
|  337|   2571|0.9957499999999999|dystopic future|
|  890|   2571|           0.99275|          scifi|
|  280|   2571|            0.9915|      cyberpunk|
|  886|   2571|0.9884999999999999|         sci fi|
|  889|   2571|            0.9875|science fiction|
+-----+-------+------------------+---------------+



## Atividade Turma

### Quais são as top 5 tags mais relevantes segundo os dados do genoma de tags dos top 3 filmes mais avaliados?

In [ ]:
%%time

df_task = ratings_df.groupBy('movieId')\
  .count()\
  .withColumnRenamed('count', 'ratings_count')\
  .orderBy(F.desc('ratings_count'))\
  .limit(3)\
  .cache()
df_task.toPandas()

CPU times: user 19.1 ms, sys: 1.52 ms, total: 20.6 ms
Wall time: 28.9 s


,movieId,ratings_count
0,356,81491
1,318,81482
2,296,79672


In [ ]:
movies_tag_gnome_df = movies_df.select('movieId', 'title')\
  .join(tag_genome_df, on='movieId', how='inner')
movies_tag_gnome_df.limit(3).toPandas()

,movieId,title,tagId,relevance,tag
0,1,Toy Story (1995),1,0.028750,007
1,1,Toy Story (1995),2,0.023750,007 (series)
2,1,Toy Story (1995),3,0.062500,18th century


In [ ]:
top_3_movies = df_task.toPandas()['movieId'].tolist()
top_3_movies

[356, 318, 296]

In [ ]:
%%time

from pyspark.sql import Window

window = Window.partitionBy('movieId').orderBy(F.desc('relevance'))

movies_tag_gnome_df.where(F.col('movieId').isin(top_3_movies))\
  .withColumn('row_number', F.row_number().over(window))\
  .where(F.col('row_number') <= 5)\
  .toPandas()

CPU times: user 11.8 ms, sys: 5.34 ms, total: 17.2 ms
Wall time: 360 ms


,movieId,title,tagId,relevance,tag,row_number
0,296,Pulp Fiction (1994),510,0.999250,hit men,1
1,296,Pulp Fiction (1994),463,0.990500,gratuitous violence,2
2,296,Pulp Fiction (1994),289,0.984500,dark humor,3
3,296,Pulp Fiction (1994),634,0.983500,masterpiece,4
4,296,Pulp Fiction (1994),536,0.983250,imdb top 250,5
5,318,"Shawshank Redemption, The (1994)",536,0.984250,imdb top 250,1
6,318,"Shawshank Redemption, The (1994)",756,0.980750,oscar (best picture),2
7,318,"Shawshank Redemption, The (1994)",806,0.977500,powerful ending,3
8,318,"Shawshank Redemption, The (1994)",634,0.973500,masterpiece,4
9,318,"Shawshank Redemption, The (1994)",813,0.972250,prison,5


In [ ]:
# Diferença entre rank, dense_rank e row_number. Observe o que acontece com valores repetidos

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# Exemplo simples
data = [
    ("A", 10),
    ("B", 20),
    ("C", 20),
    ("C", 20),
    ("C", 20),
    ("D", 30)
]

df = spark.createDataFrame(data, ["nome", "valor"])

w = Window.orderBy(F.desc("valor"))

df = df.select(
    "nome",
    "valor",
    F.rank().over(w).alias("rank"),
    F.dense_rank().over(w).alias("dense_rank"),
    F.row_number().over(w).alias("row_number")
)

df.show()

+----+-----+----+----------+----------+
|nome|valor|rank|dense_rank|row_number|
+----+-----+----+----------+----------+
|   D|   30|   1|         1|         1|
|   B|   20|   2|         2|         2|
|   C|   20|   2|         2|         3|
|   C|   20|   2|         2|         4|
|   C|   20|   2|         2|         5|
|   A|   10|   6|         3|         6|
+----+-----+----+----------+----------+

